<a href="https://colab.research.google.com/github/pmetheny99/hello-world/blob/main/PNC_1_20_titoli_ultimo_anno_ver2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# =============================================================================
# NOME SCRIPT: nexi_pnc_analisi.py
# VERSIONE: 2.5.3 (Revisione del 17-04-2024)
# DESCRIZIONE: Analisi YTD PNC con focus su PNC Attuale (con data) e PNC MAX.
# REVISIONI:
# - v2.5.2: Integrazione valore PNC MAX YTD.
# - v2.5.3: Aggiunta opzione TITOLO_SPECIFICO e riordino colonne console.
# =============================================================================

import os
import subprocess
import sys
import time

# --- PARAMETRI DI CONFIGURAZIONE ---
NUMERO_TITOLI_DA_ANALIZZARE = 4
# Se vuoi analizzare un solo titolo specifico, inserisci il nome esatto qui sotto (es: "NEXI SPA")
# Se lasciato vuoto "", lo script userà il parametro NUMERO_TITOLI_DA_ANALIZZARE.
TITOLO_SPECIFICO = ""
# -----------------------------------

def setup_environment():
    """Installa le librerie necessarie se mancano."""
    packages = ['openpyxl', 'plotly', 'pandas']
    for p in packages:
        try:
            __import__(p)
        except ImportError:
            print(f"📦 Installazione libreria: {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", p])

    try:
        import kaleido
    except ImportError:
        print("📦 Installazione motore grafico (Kaleido)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "kaleido==0.1.0.post1"])

setup_environment()

import pandas as pd
import plotly.graph_objects as go
import zipfile
from datetime import datetime
from google.colab import files

def analizza_pnc_ytd(n_top_titoli=5, titolo_target=""):
    print(f"📊 Avvio analisi PNC YTD v2.5.3 (Update: {datetime.now().strftime('%d/%m/%Y')})")

    # 1. Rilevamento file
    all_files = os.listdir()
    excel_files = [f for f in all_files if 'PncPubbl' in f and f.endswith('.xlsx')]

    if not excel_files:
        print("\n❌ ERRORE: Caricare il file 'PncPubbl.xlsx' su Colab prima di avviare!")
        return

    f_excel = excel_files[0]

    # 2. Caricamento Dati
    try:
        xl = pd.ExcelFile(f_excel)
        sheet_curr = [s for s in xl.sheet_names if 'Correnti' in s or 'Current' in s][0]
        sheet_hist = [s for s in xl.sheet_names if 'Storiche' in s or 'Historic' in s][0]
        df_curr = pd.read_excel(f_excel, sheet_name=sheet_curr)
        df_hist = pd.read_excel(f_excel, sheet_name=sheet_hist)
    except Exception as e:
        print(f"❌ Errore caricamento Excel: {e}")
        return

    col_perc = 'Perc. posizione netta corta'
    col_data = 'Data della posizione'
    col_emittente = 'Emittente'
    col_detentore = 'Detentore'

    def clean_df(df):
        df[col_perc] = pd.to_numeric(df[col_perc], errors='coerce')
        df[col_data] = pd.to_datetime(df[col_data], dayfirst=True, errors='coerce', format='mixed')
        return df.dropna(subset=[col_perc, col_data])

    df_curr, df_hist = clean_df(df_curr), clean_df(df_hist)
    df_all = pd.concat([df_curr, df_hist], ignore_index=True)

    # 3. Selezione Titoli (Logica Priorità)
    if titolo_target and titolo_target.strip() != "":
        if titolo_target in df_all[col_emittente].unique():
            print(f"🎯 Focus prioritario su titolo singolo: {titolo_target}")
            top_titoli = [titolo_target]
        else:
            print(f"⚠️ Titolo '{titolo_target}' non trovato. Procedo con la Top {n_top_titoli}.")
            top_titoli = df_curr.groupby(col_emittente)[col_perc].sum().sort_values(ascending=False).head(n_top_titoli).index.tolist()
    else:
        print(f"🔎 Analisi focalizzata sui primi {n_top_titoli} titoli per PNC complessiva.")
        top_titoli = df_curr.groupby(col_emittente)[col_perc].sum().sort_values(ascending=False).head(n_top_titoli).index.tolist()

    # 4. Timeline YTD
    start_ytd = pd.Timestamp(year=datetime.now().year, month=1, day=1)
    end_ytd = pd.Timestamp.now().normalize()
    date_range = pd.date_range(start=start_ytd, end=end_ytd, freq='B')

    fig_main = go.Figure()
    fig_rec = go.Figure()

    colors = [
        '#32CD32', '#FF4500', '#1E90FF', '#FFD700', '#FF00FF', '#00FFFF', '#FFA500', '#ADFF2F',
        '#FF69B4', '#8A2BE2', '#00FF7F', '#F4A460', '#D2691E', '#7FFF00', '#DC143C', '#00CED1',
        '#9400D3', '#FFDAB9', '#00FA9A', '#B0C4DE'
    ]

    all_data_log = []
    summary_list = []

    for i, emittente in enumerate(top_titoli):
        df_t = df_all[df_all[col_emittente] == emittente].copy()
        color = colors[i % len(colors)]
        var_dates = df_t[col_data].dt.normalize().unique()

        daily_pnc = []
        for d in date_range:
            snap = df_t[df_t[col_data] <= d]
            if snap.empty:
                daily_pnc.append({'Data': d, 'PNC': 0.0, 'Var': False})
                continue
            val = snap.sort_values(col_data).groupby(col_detentore).tail(1)
            total = val[val[col_perc] >= 0.5][col_perc].sum()
            daily_pnc.append({'Data': d, 'PNC': total, 'Var': d in var_dates})

        df_d = pd.DataFrame(daily_pnc)

        # Calcolo Metriche
        curr_val = df_d['PNC'].iloc[-1]
        pnc_max = df_d['PNC'].max()

        # Logica Recovery (PNC attuale < 80% del massimo)
        recovery = (curr_val < pnc_max * 0.8)

        # Aggiunta riga sintesi con ordine richiesto
        summary_list.append({
            'Emittente': emittente,
            'PNC Attuale %': round(curr_val, 3),
            'Data Ultima': df_d['Data'].iloc[-1].strftime('%d/%m'),
            'PNC Max %': round(pnc_max, 3)
        })

        # Plot Principale
        fig_main.add_trace(go.Scatter(
            x=df_d['Data'], y=df_d['PNC'], name=emittente, mode='lines+markers',
            marker=dict(size=[7 if v else 0 for v in df_d['Var']], color=color),
            line=dict(color=color, width=2),
            customdata=[pnc_max]*len(df_d),
            text=[emittente]*len(df_d),
            hovertemplate="<b>%{text}</b><br>Data: %{x|%d %b}<br>PNC: %{y:.2f}%<br>Max YTD: %{customdata:.2f}%<extra></extra>"
        ))

        if recovery:
            fig_rec.add_trace(go.Scatter(
                x=df_d['Data'], y=df_d['PNC'], name=emittente, mode='lines+markers',
                marker=dict(size=[5 if v else 0 for v in df_d['Var']], color=color),
                line=dict(color=color, width=1.5, dash='dot'),
                text=[emittente]*len(df_d),
                hovertemplate="<b>%{text}</b><br>Data: %{x|%d %b}<br>PNC: %{y:.2f}%"
            ))

        df_d['Emittente'] = emittente
        df_d['PNC_Max_YTD'] = pnc_max
        all_data_log.append(df_d)

    # Layout
    lt = dict(
        template="plotly_dark", height=500, width=1000,
        xaxis=dict(rangebreaks=[dict(bounds=["sat", "mon"])], tickformat="%d %b", title="Linea Temporale"),
        yaxis=dict(title="PNC Totale %"),
        legend=dict(font=dict(size=9), orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=50, r=150, t=50, b=50)
    )

    fig_main.update_layout(title=f"1. Evoluzione PNC Totali (YTD) - {len(top_titoli)} titoli", **lt)
    fig_rec.update_layout(title="2. Segnali di Riduzione (Recovery)", **lt)

    print("\n📊 Riepilogo Dati YTD:")
    print(pd.DataFrame(summary_list).to_string(index=False))

    fig_main.show()

    # ESPORTAZIONE
    print("\n📦 Generazione file di export...")
    fig_main.write_html("1_PNC_Totale.html")
    fig_rec.write_html("2_Short_Covering.html")

    try:
        import kaleido
        fig_main.write_image("1_PNC_Totale.png", engine="kaleido")
        fig_rec.write_image("2_Short_Covering.png", engine="kaleido")
        print("✅ Immagini PNG salvate.")
    except:
        print("⚠️ Nota: PNG non generati. HTML e Excel disponibili.")

    # Excel
    xlsx_file = f"Report_PNC_Analisi_v2.5.3.xlsx"
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        pd.concat(all_data_log).to_excel(writer, sheet_name='Storico_Dettagliato', index=False)
        pd.DataFrame(summary_list).to_excel(writer, sheet_name='Sintesi', index=False)
        for sn in writer.sheets:
            ws = writer.sheets[sn]
            for col in ws.columns:
                val_len = [len(str(cell.value)) for cell in col if cell.value is not None]
                if val_len:
                    ws.column_dimensions[col[0].column_letter].width = max(val_len) + 3

    # ZIP Final
    zip_fn = "Analisi_PNC_v2.5.3_Completa.zip"
    with zipfile.ZipFile(zip_fn, 'w') as z:
        for f in ["1_PNC_Totale.html", "2_Short_Covering.html", xlsx_file, "1_PNC_Totale.png", "2_Short_Covering.png"]:
            if os.path.exists(f): z.write(f)

    print(f"\n✅ ZIP pronto per il download: {zip_fn}")
    files.download(zip_fn)

# Esecuzione
analizza_pnc_ytd(n_top_titoli=NUMERO_TITOLI_DA_ANALIZZARE, titolo_target=TITOLO_SPECIFICO)

📊 Avvio analisi PNC YTD v2.5.3 (Update: 17/04/2026)


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning:

Workbook contains no default style, apply openpyxl's default

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning:

Workbook contains no default style, apply openpyxl's default



🔎 Analisi focalizzata sui primi 4 titoli per PNC complessiva.

📊 Riepilogo Dati YTD:
   Emittente  PNC Attuale % Data Ultima  PNC Max %
  SAIPEM SPA          10.69       17/04      13.14
BFF BANK SPA           6.25       17/04       6.25
    NEXI SPA           5.91       17/04       7.89
    AVIO SPA           4.25       17/04       4.61



📦 Generazione file di export...
⚠️ Nota: PNG non generati. HTML e Excel disponibili.

✅ ZIP pronto per il download: Analisi_PNC_v2.5.3_Completa.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>